# Notebook ١: Pandas وNumPy على داتا الكورة

داتا Fantasy Premier League الحقيقية من الـ API الرسمي، لقطة يوم 26 نوفمبر 2026. كل Output اتشغل فعلًا.

## ١. نجيب الداتا ونحطها في DataFrame

كل سطر كود تحته شرح بسيط لوظيفته.

In [ ]:
# نستورد مكتبة json عشان نقرأ ملفات JSON
import json
# نستورد pandas للجداول باسم مختصر pd
import pandas as pd
# نستورد numpy للحسابات على المصفوفات
import numpy as np

# السطر ده بيفتح ملف الـ JSON
with open("fpl.json") as f:
# بنحوّل محتوى الملف لـ dict في بايثون
    raw = json.load(f)

# جدول اللاعبين من مفتاح elements
players = pd.DataFrame(raw["elements"])
# جدول الأندية من مفتاح teams
teams = pd.DataFrame(raw["teams"])
# نطبع (عدد الصفوف، عدد الأعمدة)
print(players.shape)
# نطبع أول 6 مفاتيح في الملف
print(list(raw.keys())[:6])


In [ ]:
# نحدد الأعمدة اللي محتاجينها
cols = ["web_name", "team", "element_type", "now_cost",
# كمّل القايمة: نقاط ودقايق وأهداف
        "total_points", "minutes", "goals_scored",
# وآخر أعمدة: التمريرات الحاسمة والأخبار
        "assists", "news"]
# ناخد الأعمدة دي بس في نسخة مستقلة
df = players[cols].copy()
# نعرض أول 5 صفوف من 4 أعمدة
df[["web_name", "team", "now_cost", "total_points"]].head(5)


## ٢. ننضف ونفهم الأعمدة

الأعمدة جاية كأرقام (team=1)، فنحولها لأسماء بـ map.

In [ ]:
# قاموس من رقم النادي لاسمه
team_name = dict(zip(teams["id"], teams["name"]))
# قاموس أرقام المراكز لأسمائها
pos_name = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD"}

# نستبدل رقم النادي باسمه
df["team"] = df["team"].map(team_name)
# عمود جديد باسم المركز
df["pos"] = df["element_type"].map(pos_name)
# السعر متخزن مضروب في 10 فنقسمه
df["price"] = df["now_cost"] / 10
# نمسح الأعمدة الرقمية القديمة
df = df.drop(columns=["element_type", "now_cost"])
# نتأكد من النتيجة
df[["web_name", "team", "pos", "price"]].head(5)


## ٣. فلترة وترتيب

نفس فكرة تحليل Arsenal في الـ GitHub بتاعك.

In [ ]:
# نختار صفوف أرسنال بس
arsenal = df[df["team"] == "Arsenal"]
# نرتبهم من الأعلى نقاط للأقل
top = arsenal.sort_values("total_points", ascending=False)
# نعرض أعلى 6
top[["web_name", "pos", "total_points", "goals_scored", "assists"]].head(6)


In [ ]:
# وسط ملعب لعب 500+ دقيقة وسعره أقل من 7
mids = df[(df["pos"] == "MID") & (df["minutes"] > 500) & (df["price"] < 7.0)]
# نطبع عددهم
print(len(mids), "cheap midfielders")
# أعلى 5 نقاط منهم
mids.nlargest(5, "total_points")[["web_name", "team", "price", "total_points"]]


## ٤. groupby: ملخص لكل مركز

أهم أداة لعمل features: نقسم الداتا لمجموعات ونحسب لكل مجموعة.

In [ ]:
# نقسم اللاعبين حسب المركز ونجمّع
summary = df.groupby("pos").agg(
# عدد اللاعبين في كل مركز
    players=("web_name", "count"),
# متوسط السعر
    avg_price=("price", "mean"),
# متوسط النقاط
    avg_points=("total_points", "mean"),
# أعلى نقاط
    max_points=("total_points", "max"),
# نقرّب لرقمين عشريين
).round(2)
# نعرض الجدول
summary


## ٥. NumPy: points per 90

vectorization: العملية بتتطبق على العمود كله من غير loop.

In [ ]:
# نحوّل عمود الدقايق لمصفوفة NumPy
played = df["minutes"].to_numpy()
# نفس الشيء للنقاط
points = df["total_points"].to_numpy()
# نقاط لكل 90 دقيقة، وصفر لو ما لعبش
p90 = np.where(played > 0, points / np.maximum(played, 1) * 90, 0.0)
# نضيفها كعمود جديد
df["pts_per90"] = p90.round(2)
# نبقي على اللي لعب 900 دقيقة أو أكتر
reg = df[df["minutes"] >= 900]
# أعلى 5
reg.nlargest(5, "pts_per90")[["web_name", "team", "minutes", "pts_per90"]]


In [ ]:
# نجمّع القيم حسب المركز
g = reg.groupby("pos")["pts_per90"]
# z-score جوه كل مركز
z = (reg["pts_per90"] - g.transform("mean")) / g.transform("std")
# نضيف عمود z
reg = reg.assign(z=z.round(2))
# أعلى 5 z-score
reg.nlargest(5, "z")[["web_name", "pos", "pts_per90", "z"]]


## ٦. Matplotlib

السعر قصاد النقاط.

In [ ]:
# مكتبة الرسم
import matplotlib.pyplot as plt
# اللاعبين اللي لعبوا كتير بس
d = df[df["minutes"] >= 900]
# نعمل لوحة رسم بمقاس 5×4
plt.figure(figsize=(5, 4))
# لكل مركز ولون
for p, col in [("DEF", "tab:blue"), ("MID", "tab:green"), ("FWD", "tab:red")]:
# نختار لاعبين المركز ده
    s = d[d["pos"] == p]
# نرسم نقطة لكل لاعب
    plt.scatter(s["price"], s["total_points"], s=12, c=col, label=p)
# اسم محور السعر
plt.xlabel("price")
# اسم محور النقاط
plt.ylabel("total points")
# نعرض مفتاح الألوان
plt.legend()
# عنوان الرسمة
plt.title("Price vs points")
# نعرض الرسمة
plt.show()


## ٧. جسر لـ NLP: عمود news

عمود نصي حقيقي فيه الإصابات والانتقالات.

In [ ]:
# نختار الصفوف اللي فيها نص أخبار
news = df[df["news"].str.len() > 0]["news"]
# عددهم
print(len(news), "players with news")
# 4 أمثلة عشوائية ثابتة
news.sample(4, random_state=1).tolist()


In [ ]:
# عدّاد جاهز للكلمات
from collections import Counter
# مكتبة التعبيرات النمطية
import re

# قايمة فاضية للكلمات
words = []
# نلف على كل خبر
for t in news:
# نصغّر الحروف ونستخرج الكلمات
    words += re.findall(r"[a-z]+", t.lower())
# أكتر 8 كلمات تكرارًا
Counter(words).most_common(8)


In [ ]:
# دالة بتصنف نوع الخبر
def news_type(t):
# نصغّر الحروف
    t = t.lower()
# لو فيه إيقاف
    if "suspend" in t: return "suspended"
# لو إصابة أو ضربة
    if "injur" in t or "knock" in t: return "injury"
# لو إعارة أو رحيل
    if "loan" in t or "left" in t: return "transfer"
# غير كده
    return "other"

# نطبقها على كل صف
df["news_type"] = df["news"].apply(lambda t: news_type(t) if t else "none")
# نعد كل نوع
df["news_type"].value_counts()
